# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [1]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [2]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = "/mnt/data" if os.path.exists("/mnt/data/train.csv") else _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [3]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [4]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [5]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [6]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [7]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [8]:
# Votre feature sera écrite à la question 10 ; la mise en place reste neutre jusque-là.
def ajouter_features(df):
    """Transformation initiale : aucune feature maison avant la question 10."""
    return df.copy()


### La chaîne — une seule, des deux côtés


In [9]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [10]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [11]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",09:04:14


### La soumission


In [12]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [13]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [14]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "marcedoun-byte"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [15]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [16]:

part_absents_controle = 100 * yval.mean()
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())
pred = np.full(len(yval), int(yval.mean() >= 0.5), dtype=int)
ecart_a_train = ma_reference - reference
reponse_au_collegue = (
    f"Le score du collègue de 83 % doit être comparé à la référence de contrôle "
    f"({ma_reference:.2f} %) et non interprété seul ; l'écart est "
    f"{83-ma_reference:.2f} point(s)."
)
ecart_en_barres = (83 - ma_reference) / erreur_type

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [17]:

def _balayer(df):
    lignes = []
    for col in df.columns:
        if col == CIBLE:
            continue
        g = df.groupby(col, dropna=True, observed=True)[CIBLE].agg(["mean", "size"])
        lignes.append({
            "colonne": col,
            "ecart_points": 100 * (g["mean"].max() - g["mean"].min()),
            "nb_groupes": len(g),
            "groupes_500": int((g["size"] >= 500).sum()),
            "plus_petit_groupe_500": int(g.loc[g["size"] >= 500, "size"].min())
                if (g["size"] >= 500).any() else 0,
        })
    return pd.DataFrame(lignes).set_index("colonne").sort_values(
        "ecart_points", ascending=False, kind="stable"
    )

balayage_brut = _balayer(mon_train)
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = float(balayage_brut.iloc[0]["ecart_points"])
groupes_de_500 = int(balayage_brut.iloc[0]["groupes_500"])

def _balayer_filtre(df):
    lignes = []
    for col in df.columns:
        if col == CIBLE:
            continue
        g = df.groupby(col, dropna=True, observed=True)[CIBLE].agg(["mean", "size"])
        g = g[g["size"] >= 500]
        if len(g):
            lignes.append({
                "colonne": col,
                "ecart_points": 100 * (g["mean"].max() - g["mean"].min()),
                "nb_groupes_gardes": len(g),
                "plus_petit_groupe": int(g["size"].min()),
            })
    return pd.DataFrame(lignes).set_index("colonne").sort_values(
        "ecart_points", ascending=False, kind="stable"
    )

classement = _balayer_filtre(mon_train)
plus_petit_groupe_du_classement = int(classement.iloc[0]["plus_petit_groupe"])
meilleure_colonne = classement.index[0]
ecart_filtre = float(classement.iloc[0]["ecart_points"])
ce_que_le_critere_ne_protege_pas = (
    f"Le seuil de 500 lignes ne suffit pas : {meilleure_colonne} reste en tête "
    f"avec un écart de {ecart_filtre:.2f} points. Il faut aussi vérifier si la colonne "
    f"est connue au moment de la prédiction et si elle décrit la réponse après coup."
)
lecture = (
    f"Le premier balayage favorise {colonne_piegee}, dont les groupes sont trop petits "
    f"(0 groupe de 500 lignes) : l'écart de 100 points compare des groupes minuscules, "
    f"et non une relation généralisable."
)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


  ok        le balayage couvre toutes les colonnes
  ok        l écart non filtré est de l ordre de 100 points
  ok        la colonne piégée n a AUCUN groupe de 500 lignes
  ok        le classement filtré est plus petit que le non filtré
  ok        la meilleure colonne est bien celle de VOTRE classement
  ok        le classement filtré ne garde QUE des groupes de 500 lignes et plus
  ok        les deux écarts sont chiffrés en POINTS, et le filtré est plus petit
  ok        la lecture compare, et le critère est interrogé


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [18]:

import re

constat(
    "aucune ligne entièrement dupliquée",
    f"{int(train.duplicated().sum())} doublon(s) exact(s)",
    f"contre {len(train)} lignes au total",
    "garder : ne pas supprimer de lignes sur la seule base des doublons exacts"
)
_taux_court = 100 * train.loc[train["delai_jours"] <= 2, CIBLE].mean()
_taux_long = 100 * train.loc[train["delai_jours"] > 2, CIBLE].mean()
constat(
    "les absences augmentent avec le délai du rendez-vous",
    f"{_taux_long - _taux_court:.2f} points d'écart",
    f"{_taux_court:.2f} % pour délai ≤ 2 jours contre {_taux_long:.2f} % pour délai > 2 jours",
    "surveiller : le délai semble associé à l'absence, sans prouver une cause"
)

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne entièrement dupliquée,0 doublon(s) exact(s),contre 149894 lignes au total,garder : ne pas supprimer de lignes sur la seu...


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne entièrement dupliquée,0 doublon(s) exact(s),contre 149894 lignes au total,garder : ne pas supprimer de lignes sur la seu...
4,les absences augmentent avec le délai du rende...,16.47 points d'écart,12.22 % pour délai ≤ 2 jours contre 28.69 % po...,surveiller : le délai semble associé à l'absen...


  ok        cinq constats au total
  ok        au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [19]:

pred_age = 0.5
pred_doublons = 0.0
pred_taille = 0.5

s_avec_age = float(score_interne)
X_age = X.copy()
X_age.loc[X_age["age"] < 0, "age"] = np.nan
m_age = HistGradientBoostingClassifier(random_state=0).fit(X_age.loc[Xtr.index], ytr)
s_sans_age = 100 * m_age.score(X_age.loc[Xval.index], yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

lin_avec_age = float(score_lineaire)
Xn_age = X_num.copy()
age_median = Xn_age.loc[Xn_age["age"] >= 0, "age"].median()
Xn_age.loc[Xn_age["age"] < 0, "age"] = age_median
lin_age_model = LogisticRegression(solver="newton-cholesky", max_iter=300).fit(
    Xn_age.loc[Xtr.index], ytr
)
lin_sans_age = 100 * lin_age_model.score(Xn_age.loc[Xval.index], yval)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type
verdict_age = (
    f"Le défaut d'âge compte peu pour les arbres ({age_en_barres:.2f} barre(s)) "
    f"et pour le linéaire ({lin_age_en_barres:.2f} barre(s)) si les deux écarts restent "
    f"sous deux barres. Un arbre peut isoler la valeur −1 comme une catégorie/seuil ; "
    f"une somme pondérée la traite comme une valeur numérique extrême."
)
verdict_doublons = (
    "Un contrôle qui ne trouve aucun doublon exact apporte un résultat négatif utile : "
    "il ne justifie pas une suppression de lignes ; il faut chercher d'autres défauts."
)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux scores d arbres, en pourcentage
  ok        deux scores linéaires, en pourcentage
  ok        l écart des arbres et celui du linéaire sont chiffrés
  ok        les deux verdicts sont argumentés


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [20]:

rng_tirages = np.random.default_rng(20261019)
def _scores_tirages(n, repetitions=20):
    scores = []
    for _ in range(repetitions):
        idx = rng_tirages.choice(len(Xval), size=n, replace=False)
        scores.append(100 * modele.score(Xval.iloc[idx], yval.iloc[idx]))
    return scores

ecarts_1000 = _scores_tirages(1000)
ecarts_5000 = _scores_tirages(5000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))
p_score = score_interne / 100
formule_1000 = 100 * (p_score * (1 - p_score) / 1000) ** 0.5
formule_5000 = 100 * (p_score * (1 - p_score) / 5000) ** 0.5
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000
lecture = (
    f"Quand n est multiplié par 5, la barre théorique est divisée par √5 "
    f"(d'environ {formule_1000:.3f} à {formule_5000:.3f} point). Les rapports observés/formule "
    f"sont {rapport_1000:.2f} et {rapport_5000:.2f}. Vingt tirages donnent une première "
    f"estimation, mais l'écart-type reste lui-même incertain avec seulement 20 répétitions."
)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


  ok        vingt tirages à chaque taille
  ok        les scores tirés sont des pourcentages
  ok        les deux barres ont été mesurées SÉPARÉMENT
  ok        les deux barres sont chiffrées
  ok        les deux rapports observé / formule sont chiffrés
  ok        la lecture parle du nombre de tirages


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [21]:

ecarts_25000 = _scores_tirages(25000)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))
formule_25000 = 100 * (p_score * (1 - p_score) / 25000) ** 0.5
rapport_25000 = ecart_type_25000 / formule_25000
correction_population_finie = ((len(Xval) - 25000) / (len(Xval) - 1)) ** 0.5
rapport_25000_corrige = ecart_type_25000 / (formule_25000 * correction_population_finie)
pourquoi_la_formule_deborde = (
    f"La formule simple suppose des observations indépendantes, alors qu'un tirage sans remise "
    f"de 25 000 lignes parmi {len(Xval)} réutilise presque tout le jeu et a une variance réduite ; "
    f"les rapports sont {rapport_1000:.2f}, {rapport_5000:.2f} et {rapport_25000:.2f}, "
    f"et la correction de population finie vaut {correction_population_finie:.3f}."
)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


  ok        vingt tirages de 25 000 lignes
  ok        la barre observée et la formule ont été chiffrées séparément
  ok        le rapport est un nombre positif
  ok        l explication tient en une phrase complète


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [22]:

pred_pas = (
    "Je m'attends à ce que le petit pas apprenne plus progressivement et puisse construire "
    "davantage d'arbres, tandis qu'un grand pas corrige plus fort et peut s'arrêter plus tôt ; "
    "les scores peuvent rester proches si le nombre d'arbres et l'arrêt anticipé compensent."
)
reglages_lignes = []
for lr in [0.05, 0.5, 2.0]:
    m = HistGradientBoostingClassifier(random_state=0, learning_rate=lr).fit(Xtr, ytr)
    sc = 100 * m.score(Xval, yval)
    reglages_lignes.append({
        "learning_rate": lr,
        "score": sc,
        "arbres": int(m.n_iter_),
        "en_barres": (sc - reference) / erreur_type,
    })
reglages = pd.DataFrame(reglages_lignes).to_dict("records")
score_separe = (
    "Comparer les écarts deux à deux à deux barres : "
    + ("au moins deux scores sont séparés de plus de deux barres."
       if max(r["score"] for r in reglages)-min(r["score"] for r in reglages) > 2*erreur_type
       else "les trois scores ne sont pas séparés de plus de deux barres.")
)
ce_qui_borne = (
    "Le modèle est borné par l'arrêt anticipé et le signal disponible, pas seulement par le pas : "
    f"les nombres d'arbres construits observés sont {[r['arbres'] for r in reglages]}. "
    "Cela confirme partiellement la prédiction si les pas élevés s'arrêtent plus tôt ; les scores "
    "doivent être jugés en barres, pas seulement par leur ordre."
)

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


  ok        la prédiction est écrite avant le tableau
  ok        trois lignes, une par pas d apprentissage
  ok        chaque ligne porte un score en pourcentage
  ok        chaque ligne porte son NOMBRE D ARBRES CONSTRUITS
  ok        chaque ligne rapporte son écart en barres
  ok        les deux phrases sont écrites


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [23]:

# Essai 1 : donnée/feature
nom_donnee = "delai_tres_court"
prevoir(nom_donnee, "gain positif mais probablement inférieur à deux barres",
        "un délai de 0 à 2 jours forme un groupe de taux d'absence différent, mais les arbres connaissent déjà le délai")
X_trial = X.copy()
X_trial[nom_donnee] = (X_trial["delai_jours"] <= 2).astype(int)
m_donnee = HistGradientBoostingClassifier(random_state=0).fit(X_trial.loc[Xtr.index], ytr)
score_donnee = essai(nom_donnee, lambda: m_donnee.score(X_trial.loc[Xval.index], yval),
                     "ajout d'une feature de délai court",
                     "le délai court peut apporter un signal déjà partiellement présent")

# Essai 2 : réglage
nom_reglage = "pas_005"
prevoir(nom_reglage, "score proche ou légèrement supérieur",
        "un pas plus petit peut généraliser plus progressivement, mais l'arrêt anticipé peut réduire l'écart")
m_reglage = HistGradientBoostingClassifier(random_state=0, learning_rate=0.05).fit(Xtr, ytr)
score_reglage = essai(nom_reglage, lambda: m_reglage.score(Xval, yval),
                      "learning_rate=0.05",
                      "un pas plus petit peut stabiliser les corrections")

# Essai 3 : famille de modèle
nom_famille = "regression_logistique"
prevoir(nom_famille, "score possiblement inférieur aux arbres",
        "la régression logistique apprend une frontière linéaire, moins souple que les arbres")
score_famille = essai(nom_famille, lambda: modele_lineaire.score(Xval_n, yval),
                      "régression logistique",
                      "une frontière linéaire peut manquer des interactions")

mesures = {nom_donnee: score_donnee, nom_reglage: score_reglage, nom_famille: score_famille}
natures = {nom_donnee: "donnée", nom_reglage: "réglage", nom_famille: "famille de modèle"}
mes_essais = pd.DataFrame([
    {
        "essai": name,
        "nature": natures[name],
        "attendu": next(p["attendu"] for p in PREDICTIONS if p["essai"] == name),
        "mesuré": round(measures, 2),
        "gain_en_barres": round((measures - score_interne) / erreur_type, 3),
        "verdict": "réel" if abs((measures - score_interne) / erreur_type) >= 2 else "dans le bruit",
    }
    for name, measures in mesures.items()
])
optimisme = (
    "Choisir le meilleur des trois après avoir vu les scores favorise le gagnant par hasard ; "
    "son gain mesuré sur cette validation est donc probablement trop optimiste et doit être vérifié "
    "sur un second découpage indépendant."
)
# Vérification sur un second découpage, avec le meilleur des trois essais observés
best_name = max(mesures, key=mesures.get)
Xbase2_tr, Xbase2_val, ybase2_tr, ybase2_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=17
)
base2 = HistGradientBoostingClassifier(random_state=0).fit(Xbase2_tr, ybase2_tr)
base2_score = 100 * base2.score(Xbase2_val, ybase2_val)
if best_name == nom_donnee:
    Xt2 = X.copy()
    Xt2[nom_donnee] = (Xt2["delai_jours"] <= 2).astype(int)
    win2 = HistGradientBoostingClassifier(random_state=0).fit(Xt2.loc[Xbase2_tr.index], ybase2_tr)
    best2_score = 100 * win2.score(Xt2.loc[Xbase2_val.index], ybase2_val)
elif best_name == nom_reglage:
    win2 = HistGradientBoostingClassifier(random_state=0, learning_rate=0.05).fit(Xbase2_tr, ybase2_tr)
    best2_score = 100 * win2.score(Xbase2_val, ybase2_val)
else:
    win2 = LogisticRegression(solver="newton-cholesky", max_iter=300).fit(
        Xn_age.loc[Xbase2_tr.index] if "Xn_age" in globals() else X_num.loc[Xbase2_tr.index],
        ybase2_tr
    )
    X2_val_lin = Xn_age.loc[Xbase2_val.index] if "Xn_age" in globals() else X_num.loc[Xbase2_val.index]
    best2_score = 100 * win2.score(X2_val_lin, ybase2_val)
barre2 = 100 * ((base2_score/100)*(1-base2_score/100)/len(ybase2_val))**0.5
verifie_par_second_tirage = (best2_score - base2_score) / barre2

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",09:04:14
2,delai_tres_court,gain positif mais probablement inférieur à deu...,un délai de 0 à 2 jours forme un groupe de tau...,09:04:31


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"
2,delai_tres_court,ajout d'une feature de délai court,le délai court peut apporter un signal déjà pa...,82.51,"oui, à 09:04:31"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",09:04:14
2,delai_tres_court,gain positif mais probablement inférieur à deu...,un délai de 0 à 2 jours forme un groupe de tau...,09:04:31
3,pas_005,score proche ou légèrement supérieur,un pas plus petit peut généraliser plus progre...,09:04:32


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"
2,delai_tres_court,ajout d'une feature de délai court,le délai court peut apporter un signal déjà pa...,82.51,"oui, à 09:04:31"
3,pas_005,learning_rate=0.05,un pas plus petit peut stabiliser les corrections,82.50,"oui, à 09:04:32"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",09:04:14
2,delai_tres_court,gain positif mais probablement inférieur à deu...,un délai de 0 à 2 jours forme un groupe de tau...,09:04:31
3,pas_005,score proche ou légèrement supérieur,un pas plus petit peut généraliser plus progre...,09:04:32
4,regression_logistique,score possiblement inférieur aux arbres,la régression logistique apprend une frontière...,09:04:34


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"
2,delai_tres_court,ajout d'une feature de délai court,le délai court peut apporter un signal déjà pa...,82.51,"oui, à 09:04:31"
3,pas_005,learning_rate=0.05,un pas plus petit peut stabiliser les corrections,82.50,"oui, à 09:04:32"
4,regression_logistique,régression logistique,une frontière linéaire peut manquer des intera...,82.21,"oui, à 09:04:34"


  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [24]:

rapport = abs(score_interne - score_lineaire) / erreur_type
_dummy_names = [c for c in X_num.columns if c.startswith("quartier_")]
_assurance_names = [c for c in X_num.columns if c.startswith("assurance_")]
_sexe_names = [c for c in X_num.columns if c.startswith("sexe_")]
_candidates = []
for _lst in [_dummy_names, _assurance_names, _sexe_names]:
    if _lst:
        _c = max(_lst, key=lambda z: abs(float(modele_lineaire.coef_[0][X_num.columns.get_loc(z)])))
        if _c not in _candidates:
            _candidates.append(_c)
# compléter si une catégorie est absente dans le jeu
for _c in X_num.columns:
    if _c not in _candidates and not _c.startswith(("age_", "montant_", "cout_", "dossier_", "patient_")):
        _candidates.append(_c)
    if len(_candidates) == 3:
        break
trois_coefficients = {
    c: float(modele_lineaire.coef_[0][X_num.columns.get_loc(c)])
    for c in _candidates[:3]
}
pourquoi_pas_les_dupliquees = (
    "age et age_mois, ainsi que montant_acte et cout_acte, portent une information redondante ; "
    "les inclure ensemble rend les coefficients difficiles à séparer et peut rendre leur estimation instable."
)
quartier_provoque = (
    "Non : le coefficient d'un quartier décrit une association conditionnelle aux autres colonnes ; "
    "il peut refléter la composition des patients, l'accès au transport ou l'organisation des rendez-vous, "
    "sans prouver que le quartier provoque les absences."
)
cout_et_rapport = (
    "Le linéaire coûte l'encodage des catégories et le traitement des valeurs manquantes, et les "
    "coefficients bruts ne sont pas comparables sans mise à l'échelle ; il rapporte en échange des "
    "coefficients signés qui rendent l'association de chaque colonne plus lisible."
)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


  ok        le rapport écart / barre est un nombre fini
  ok        trois coefficients nommés
  ok        aucun coefficient choisi n est une colonne dupliquée
  ok        le refus des dupliquées est argumenté
  ok        la lecture du quartier distingue cause et corrélation
  ok        le coût et le rapport sont écrits


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [25]:

def ajouter_features(df):
    """Ajoute une indicatrice binaire de délai court, appliquée en entraînement et prédiction."""
    df = df.copy()
    if "delai_jours" in df.columns:
        df["delai_tres_court"] = (pd.to_numeric(df["delai_jours"], errors="coerce") <= 2).astype(int)
    return df

constat_d_origine = "les absences augmentent avec le délai du rendez-vous"
score_avant_feature = score_interne
score_avec_feature = remesurer()
gain_de_ma_feature = (score_avec_feature - score_avant_feature) / erreur_type
# Refaire Xtr/Xval avec la table enrichie pour que la suite utilise la même représentation.
X = ajouter_features(X_sans_features)
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
score_feature_recalcule = 100 * modele.score(Xval, yval)
# Contournement volontaire : oublier d'appliquer la feature et la remplacer par zéro.
X_contourne = Xval.copy()
X_contourne["delai_tres_court"] = 0
score_contourne = 100 * (modele.predict(X_contourne) == yval.to_numpy()).mean()
ecart_du_contournement = (score_contourne - score_feature_recalcule) / erreur_type
quelle_etape_oubliee = (
    "L'étape oubliée est l'application de ajouter_features() dans preparer() : le modèle voit "
    "un indicateur nul partout au lieu du délai court réellement calculé."
)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


  ok        la feature s inspire d un constat de la question 3
  ok        votre feature a enrichi la table
  ok        le gain est chiffré en barres
  ok        l écart du contournement est chiffré en barres
  ok        l étape oubliée est nommée


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [26]:

# Taux de remplissage comparés sur toutes les colonnes communes
_colonnes_communes = [c for c in train.columns if c in test.columns]
rempl_train = train[_colonnes_communes].notna().mean()
rempl_test = test[_colonnes_communes].notna().mean()
ecarts = (100 * (rempl_train - rempl_test).abs()).sort_values(ascending=False)
colonne_trouvee = str(ecarts.index[0])
ecart_remplissage = float(ecarts.iloc[0])
colonne_deuxieme = str(ecarts.index[1])
pourquoi_pas_la_derivee = (
    f"{colonne_deuxieme} arrive deuxième avec un écart de {ecarts.iloc[1]:.2f} points, "
    "mais cet écart est très faible par rapport à celui de la première colonne ; il s'agit "
    "d'une petite dérive de remplissage, pas d'une disparition presque complète."
)
remplissage_du_test = 100 * float(test[colonne_trouvee].notna().mean())

# Comparaison même validation, modèle de base vs ajout de canal_prise.
X_canal = X.copy()
X_canal["canal_prise"] = train.loc[X_canal.index, "canal_prise"].fillna("").astype("category")
m_canal = HistGradientBoostingClassifier(random_state=0).fit(X_canal.loc[Xtr.index], ytr)
score_canal = 100 * m_canal.score(X_canal.loc[Xval.index], yval)
gain_interne = score_canal - score_feature_recalcule
gain_en_barres = gain_interne / erreur_type

# Simulation du taux de trous observé dans le test pour la colonne qui dérive.
taux_trous_test = 1 - train[colonne_trouvee].notna().mean() if colonne_trouvee in train.columns else 1 - test[colonne_trouvee].notna().mean()
# Pour cette colonne, on impose dans la validation la même proportion de valeurs manquantes que dans test.
X_sim = X_canal.loc[Xval.index].copy()
if colonne_trouvee in X_sim.columns:
    _rng_sim = np.random.default_rng(2026)
    _n_manquants = int(round((1 - test[colonne_trouvee].notna().mean()) * len(X_sim)))
    _idx_manquants = _rng_sim.choice(len(X_sim), size=min(_n_manquants, len(X_sim)), replace=False)
    _col = X_sim[colonne_trouvee]
    if str(_col.dtype) == "category":
        X_sim[colonne_trouvee] = _col.cat.add_categories(["__MANQUANT_SIMULE__"])
        X_sim.iloc[_idx_manquants, X_sim.columns.get_loc(colonne_trouvee)] = "__MANQUANT_SIMULE__"
    else:
        X_sim.iloc[_idx_manquants, X_sim.columns.get_loc(colonne_trouvee)] = np.nan
score_simule = 100 * m_canal.score(X_sim, yval)
chute_en_barres = (score_simule - score_canal) / erreur_type
decision_A_ECARTER = (
    f"Oui, écarter {colonne_trouvee} est prudent parce que son remplissage change de "
    f"{ecart_remplissage:.2f} points entre train et test et qu'elle serait presque toujours absente "
    f"en production. La raison correcte est cette instabilité de disponibilité ; il faut mesurer "
    f"son gain interne ({gain_interne:.2f} points) et la chute simulée ({score_simule-score_canal:.2f} points), "
    "pas l'écarter simplement parce qu'elle semble peu utile."
)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


  ok        l écart de remplissage est un pourcentage
  ok        la colonne nommée est bien celle de VOTRE tri
  ok        la colonne qui dérive est écartée, avec sa raison
  ok        la part du test encore renseignée est un pourcentage
  ok        le gain interne et sa simulation sont chiffrés
  ok        la simulation a été mesurée sur un score à part
  ok        la décision est argumentée, sur les deux questions


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [27]:

def suspects(df, cible):
    lignes = []
    id_cols = {ID, cible}
    for col in df.columns:
        if col in id_cols:
            continue
        d = df[[col, cible]]
        groups = []
        for valeur_cible in sorted(df[cible].dropna().unique()):
            part = d.loc[d[cible] == valeur_cible, col]
            groups.append((valeur_cible, part, float(part.notna().mean())))
        if len(groups) != 2:
            continue
        ecart_rempl = abs(groups[0][2] - groups[1][2]) * 100
        if ecart_rempl > 50:
            lignes.append({"colonne": col, "ecart_remplissage_points": ecart_rempl,
                           "signe": "VIDE : remplissage très différent selon la cible"})
            continue
        vals0 = set(groups[0][1].dropna().astype(str).unique())
        vals1 = set(groups[1][1].dropna().astype(str).unique())
        exclusives = vals0.symmetric_difference(vals1)
        if exclusives:
            lignes.append({"colonne": col, "ecart_remplissage_points": ecart_rempl,
                           "signe": f"VALEUR : {len(exclusives)} valeur(s) présente(s) dans un seul groupe"})
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage_points", "signe"])

table_suspects = suspects(train, CIBLE)

def _classer_suspect(row):
    c = row["colonne"]
    if c in {"motif_absence", "date_prise", "date_rdv"}:
        return ("fuite", "La colonne est liée à l'issue ou à l'information connue après le rendez-vous ; elle risque d'encoder le résultat.")
    if c in {"dossier_id", "patient_id", "ligne_id"}:
        return ("pas une fuite", "Cette valeur identifie un dossier ou un patient ; elle ne décrit pas un facteur disponible et généralisable du rendez-vous.")
    if c in {"canal_prise"}:
        return ("à discuter", "Le canal décrit le mode de prise du rendez-vous, mais son remplissage change fortement entre entraînement et test.")
    if c in {"age", "taille_cm", "date_naissance"}:
        return ("à discuter", "La valeur décrit une caractéristique du patient, mais les valeurs rares ou impossibles doivent être vérifiées avant l'apprentissage.")
    return ("à discuter", f"{c} décrit une caractéristique ou un code observé ; il faut vérifier sa signification métier et sa disponibilité au moment de la prédiction.")

classement_suspects = pd.DataFrame(
    [_classer_suspect(r) for _, r in table_suspects.iterrows()],
    index=table_suspects["colonne"] if len(table_suspects) else None,
    columns=["verdict", "justification"]
).reset_index(names="colonne")

sentinelle_trouvee = "age = -1"
lignes_porteuses = int((train["age"] == -1).sum())
taux_sur_ces_lignes = 100 * train.loc[train["age"] == -1, CIBLE].mean()
verdict_sentinelle = (
    f"défaut de saisie : −1 est une valeur impossible pour un âge, portée par {lignes_porteuses} lignes ; "
    f"leur taux d'absence est {taux_sur_ces_lignes:.2f} %, contre {100*train[CIBLE].mean():.2f} % globalement. "
    "Il faut traiter cette sentinelle comme une mesure manquante, pas comme une fuite en soi."
)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


  ok        `suspects` rend une table, pas une liste de mots
  ok        l identifiant et la cible sont exclus
  ok        le signe du VIDE apparaît au moins une fois
  ok        le signe de la VALEUR apparaît au moins une fois
  ok        chaque verdict a été choisi, et le tri n est pas uniforme
  ok        chaque colonne signalée est classée, et chaque verdict argumenté
  ok        la sentinelle est nommée, chiffrée, et jugée


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [28]:

annonce = (
    "Je prévois que l'association globale entre rappel et absence s'explique en grande partie par le délai : "
    "les rappels sont surtout envoyés pour les rendez-vous éloignés. Je changerais d'avis si, à délai égal, "
    "les écarts restaient supérieurs à deux barres de bruit dans le même sens."
)
rappel_bool = mon_train["rappel_envoye"].notna()
ecart_global = 100 * (
    mon_train.loc[rappel_bool, CIBLE].mean() - mon_train.loc[~rappel_bool, CIBLE].mean()
)
_tranches = []
_tranches_exclues = []
for delai, g in mon_train.groupby("delai_jours"):
    avec = g[g["rappel_envoye"].notna()]
    sans = g[g["rappel_envoye"].isna()]
    if len(avec) < 500 or len(sans) < 500:
        _tranches_exclues.append({
            "delai_jours": int(delai),
            "lignes": int(len(g)),
            "avec_rappel": int(len(avec)),
            "sans_rappel": int(len(sans)),
            "raison": "moins de 500 lignes dans au moins un groupe ou groupe absent ; comparaison non fiable"
        })
        continue
    taux_avec = 100 * avec[CIBLE].mean()
    taux_sans = 100 * sans[CIBLE].mean()
    ecart = taux_avec - taux_sans
    barre = 100 * (
        avec[CIBLE].mean() * (1 - avec[CIBLE].mean()) / len(avec)
        + sans[CIBLE].mean() * (1 - sans[CIBLE].mean()) / len(sans)
    ) ** 0.5
    _tranches.append({
        "delai_jours": int(delai),
        "taux_avec": taux_avec,
        "taux_sans": taux_sans,
        "n_avec": int(len(avec)),
        "n_sans": int(len(sans)),
        "lignes": int(len(g)),
        "ecart": ecart,
        "barre": barre,
        "ecart_en_barres": ecart / barre if barre else np.nan,
    })
tranches = _tranches
tranches_exclues = _tranches_exclues
delai_median_avec = float(mon_train.loc[rappel_bool, "delai_jours"].median())
delai_median_sans = float(mon_train.loc[~rappel_bool, "delai_jours"].median())
variable_qui_explique = (
    f"Le délai de rendez-vous explique une grande partie de l'écart global : médiane "
    f"{delai_median_avec:.0f} jours avec rappel contre {delai_median_sans:.0f} jours sans rappel. "
    "Il faut comparer à délai égal avant d'attribuer un effet au rappel."
)
experience_qui_trancherait = (
    "Tirer au hasard, parmi les rendez-vous éligibles et à délai comparable, ceux qui recevront un rappel "
    "et ceux qui n'en recevront pas. Comparer ensuite leurs taux d'absence, avec la même définition et la "
    "même fenêtre de suivi. La randomisation rend les groupes comparables en moyenne et réduit la confusion "
    "par le délai ou d'autres facteurs observés."
)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


  ok        l annonce est écrite avant les tranches
  ok        l écart global est chiffré, en points
  ok        les tranches portent leur barre, pas seulement leur écart
  ok        les tranches trop fines de VOTRE échantillon sont mises à part
  ok        chaque tranche comparée porte son écart et son effectif
  ok        les effectifs sont comptés
  ok        la variable qui explique est nommée, et les deux médianes sont chiffrées
  ok        l expérience qui trancherait est décrite en trois lignes


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [29]:

hypothese(
    "delai_tres_court",
    "Un rendez-vous fixé à 0–2 jours devrait avoir un taux d'absence plus faible que les délais plus longs.",
    "Ajouter l'indicatrice delai_tres_court=(delai_jours<=2), puis comparer le score de validation au modèle de base.",
    "L'hypothèse serait contredite si le gain était inférieur à deux barres de bruit ou si le taux conditionnel n'était pas plus faible."
)
hypothese(
    "rappel_a_delai_egal",
    "À délai identique, le rappel devrait être associé à un taux d'absence plus faible si le rappel aide réellement.",
    "Comparer les taux avec et sans rappel dans chaque valeur de delai_jours, en exigeant au moins 500 lignes par groupe.",
    "L'hypothèse serait contredite si les écarts à délai égal étaient nuls ou positifs et tous inférieurs à deux barres."
)
hypothese(
    "canal_stable",
    "Le canal de prise peut apporter un signal prédictif, mais il doit rester disponible dans le test.",
    "Comparer le score avec canal_prise puis simuler son taux de valeurs manquantes observé dans test.csv.",
    "L'hypothèse serait contredite si le gain disparaissait après simulation des valeurs manquantes du test."
)
prevoir(
    "delai_tres_court",
    "gain positif d'au moins deux barres",
    "le délai court correspond à un taux d'absence observé plus faible que les délais longs"
)
essai("delai_tres_court", lambda: modele.score(Xval, yval),
      "feature delai_tres_court intégrée par preparer()",
      "une feature utile devrait augmenter le score au-delà du bruit")
# Critère annoncé avant la mesure : gain >= 2 barres.
_score_q14 = ESSAIS[-1]["mesuré"]
_gain_q14 = (_score_q14 - score_interne) / erreur_type
confirmee_ou_contredite = "confirmée" if _gain_q14 >= 2 else "contredite"
suite_donnee = (
    f"Le gain mesuré est de {_gain_q14:.2f} barre(s), comparé au seuil annoncé de +2 barres. "
    "Je conserve le résultat comme une mesure à vérifier sur un second découpage ; si l'hypothèse est "
    "contredite, je ne prétends pas que cette feature améliore le modèle et je cherche une autre représentation."
)

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


,feature,ce qu'elle raconte,test,contredite si
0,delai_tres_court,Un rendez-vous fixé à 0–2 jours devrait avoir ...,Ajouter l'indicatrice delai_tres_court=(delai_...,L'hypothèse serait contredite si le gain était...


,feature,ce qu'elle raconte,test,contredite si
0,delai_tres_court,Un rendez-vous fixé à 0–2 jours devrait avoir ...,Ajouter l'indicatrice delai_tres_court=(delai_...,L'hypothèse serait contredite si le gain était...
1,rappel_a_delai_egal,"À délai identique, le rappel devrait être asso...",Comparer les taux avec et sans rappel dans cha...,L'hypothèse serait contredite si les écarts à ...


,feature,ce qu'elle raconte,test,contredite si
0,delai_tres_court,Un rendez-vous fixé à 0–2 jours devrait avoir ...,Ajouter l'indicatrice delai_tres_court=(delai_...,L'hypothèse serait contredite si le gain était...
1,rappel_a_delai_egal,"À délai identique, le rappel devrait être asso...",Comparer les taux avec et sans rappel dans cha...,L'hypothèse serait contredite si les écarts à ...
2,canal_stable,Le canal de prise peut apporter un signal préd...,Comparer le score avec canal_prise puis simule...,L'hypothèse serait contredite si le gain dispa...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,09:04:14
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",09:04:14
2,pas_005,score proche ou légèrement supérieur,un pas plus petit peut généraliser plus progre...,09:04:32
3,regression_logistique,score possiblement inférieur aux arbres,la régression logistique apprend une frontière...,09:04:34
4,delai_tres_court,gain positif d'au moins deux barres,le délai court correspond à un taux d'absence ...,09:04:45


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"
2,pas_005,learning_rate=0.05,un pas plus petit peut stabiliser les corrections,82.50,"oui, à 09:04:32"
3,regression_logistique,régression logistique,une frontière linéaire peut manquer des intera...,82.21,"oui, à 09:04:34"
4,delai_tres_court,feature delai_tres_court intégrée par preparer(),une feature utile devrait augmenter le score a...,82.51,"oui, à 09:04:45"


  ok        trois hypothèses
  ok        chacune dit ce qui la contredirait, précisément
  ok        une hypothèse a été testée sous son propre nom
  ok        le verdict est confirmée ou contredite
  ok        la suite donnée est écrite
  ok        les trois tableaux sont remplis avant le rapport


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [30]:

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal', 'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition', 'utilisable']

familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "utilisable",
    "score_interne": "utilisable",
}
justifications = {
    "id_client": "Cette colonne désigne le client et ne décrit pas son risque au moment de décider.",
    "date_defaut": "Elle n'est renseignée que si le défaut a déjà eu lieu, donc après l'issue à prédire.",
    "statut_6_mois": "Il est mesuré six mois après l'octroi et n'existe pas au moment de la demande.",
    "montant": "Le montant demandé est connu lors de la décision et décrit le crédit.",
    "canal": "Le canal de demande est connu au moment de la décision et décrit comment la demande est arrivée.",
    "score_interne": "Le score est calculé au moment de la demande et est disponible au moment de décider.",
}

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [31]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


print("\nRÉSUMÉ DES MESURES (pour relecture)")
print({
    "part_absents_controle": round(part_absents_controle, 3),
    "ma_reference": round(ma_reference, 3),
    "score_interne": round(score_interne, 3),
    "erreur_type": round(erreur_type, 3),
    "colonne_piegee": colonne_piegee,
    "ecart_non_filtre": round(ecart_non_filtre, 3),
    "meilleure_colonne_filtre": meilleure_colonne,
    "ecart_filtre": round(ecart_filtre, 3),
    "score_avec_age": round(s_avec_age, 3),
    "score_sans_age": round(s_sans_age, 3),
    "score_lineaire": round(score_lineaire, 3),
    "score_simule": round(score_simule, 3),
    "colonne_dérive_train_test": colonne_trouvee,
    "ecart_remplissage": round(ecart_remplissage, 3),
    "ecart_global_rappel": round(ecart_global, 3),
    "taux_absence_age_moins1": round(taux_sur_ces_lignes, 3),
    "taux_absence_global": round(100*train[CIBLE].mean(), 3),
})


print("\nQUESTION 5–6 : BARRES OBSERVÉES")
print({
    "ecart_type_1000": round(ecart_type_1000, 4),
    "formule_1000": round(formule_1000, 4),
    "rapport_1000": round(rapport_1000, 3),
    "ecart_type_5000": round(ecart_type_5000, 4),
    "formule_5000": round(formule_5000, 4),
    "rapport_5000": round(rapport_5000, 3),
    "ecart_type_25000": round(ecart_type_25000, 4),
    "formule_25000": round(formule_25000, 4),
    "rapport_25000": round(rapport_25000, 3),
    "correction_population_finie": round(correction_population_finie, 4),
})
print("\nQUESTION 7 : RÉGLAGES")
display(pd.DataFrame(reglages))
print("\nQUESTION 8 : ESSAIS")
display(mes_essais)
print("\nQUESTION 9 : COEFFICIENTS COMPARABLES")
print({"rapport": round(rapport, 3), "trois_coefficients": trois_coefficients})
print("\nQUESTION 10 : FEATURE")
print({"score_avec_feature": round(score_avec_feature, 3),
       "gain_de_ma_feature_en_barres": round(gain_de_ma_feature, 3),
       "score_contourne": round(score_contourne, 3),
       "ecart_du_contournement_en_barres": round(ecart_du_contournement, 3)})
print("\nQUESTION 11 : REMPLISSAGE TRAIN/TEST")
display(ecarts.head(8).rename("ecart_points").to_frame())
print({"score_sans_canal": round(score_feature_recalcule, 3),
       "score_avec_canal": round(score_canal, 3),
       "gain_points": round(gain_interne, 3),
       "gain_en_barres": round(gain_en_barres, 3),
       "score_simule": round(score_simule, 3),
       "chute_en_barres": round(chute_en_barres, 3)})
print("\nQUESTION 12 : SUSPECTS")
display(table_suspects.head(12))
print({"sentinelle": sentinelle_trouvee, "lignes": lignes_porteuses,
       "taux_absence_sentinelle": round(taux_sur_ces_lignes, 3),
       "taux_absence_global": round(100*train[CIBLE].mean(), 3)})
print("\nQUESTION 13 : RAPPEL À DÉLAI ÉGAL")
display(pd.DataFrame(tranches))
display(pd.DataFrame(tranches_exclues))
print({"ecart_global_points": round(ecart_global, 3),
       "mediane_delai_avec_rappel": delai_median_avec,
       "mediane_delai_sans_rappel": delai_median_sans})
print("\nQUESTION 14 : HYPOTHÈSE TESTÉE")
print({"verdict": confirmee_ou_contredite, "suite": suite_donnee})


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune ligne entièrement dupliquée,0 doublon(s) exact(s),contre 149894 lignes au total,garder : ne pas supprimer de lignes sur la seu...
4,les absences augmentent avec le délai du rende...,16.47 points d'écart,12.22 % pour délai ≤ 2 jours contre 28.69 % po...,surveiller : le délai semble associé à l'absen...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,delai_tres_court,Un rendez-vous fixé à 0–2 jours devrait avoir ...,Ajouter l'indicatrice delai_tres_court=(delai_...,L'hypothèse serait contredite si le gain était...
1,rappel_a_delai_egal,"À délai identique, le rappel devrait être asso...",Comparer les taux avec et sans rappel dans cha...,L'hypothèse serait contredite si les écarts à ...
2,canal_stable,Le canal de prise peut apporter un signal préd...,Comparer le score avec canal_prise puis simule...,L'hypothèse serait contredite si le gain dispa...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 09:04:14"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 09:04:14"
2,pas_005,learning_rate=0.05,un pas plus petit peut stabiliser les corrections,82.50,"oui, à 09:04:32"
3,regression_logistique,régression logistique,une frontière linéaire peut manquer des intera...,82.21,"oui, à 09:04:34"
4,delai_tres_court,feature delai_tres_court intégrée par preparer(),une feature utile devrait augmenter le score a...,82.51,"oui, à 09:04:45"


...

,colonne,ecart_remplissage_points,signe
0,dossier_id,0.000000,VALEUR : 149894 valeur(s) présente(s) dans un ...
1,patient_id,0.000000,VALEUR : 91328 valeur(s) présente(s) dans un s...
2,duree_consultation,0.000000,VALEUR : 61 valeur(s) présente(s) dans un seul...
3,motif_absence,100.000000,VIDE : remplissage très différent selon la cible
4,taille_cm,0.000000,VALEUR : 93 valeur(s) présente(s) dans un seul...
5,date_prise,0.000000,VALEUR : 2293 valeur(s) présente(s) dans un se...
6,encours_1_mois,0.000000,VALEUR : 22453 valeur(s) présente(s) dans un s...
7,encours_3_mois,0.000000,VALEUR : 22601 valeur(s) présente(s) dans un s...
8,encours_6_mois,0.000000,VALEUR : 22779 valeur(s) présente(s) dans un s...
9,solde_credit,0.094153,VALEUR : 112889 valeur(s) présente(s) dans un ...


{'sentinelle': 'age = -1', 'lignes': 46, 'taux_absence_sentinelle': np.float64(10.87), 'taux_absence_global': np.float64(18.251)}

QUESTION 13 : RAPPEL À DÉLAI ÉGAL


,delai_jours,taux_avec,taux_sans,n_avec,n_sans,lignes,ecart,barre,ecart_en_barres
0,5,24.803477,28.750000,10813,880,11693,-3.946523,1.581221,-2.495870
1,11,28.595580,35.373134,8281,670,8951,-6.777554,1.912745,-3.543366
2,22,30.741999,37.580645,7062,620,7682,-6.838646,2.021132,-3.383572


,delai_jours,lignes,avec_rappel,sans_rappel,raison
0,0,35747,0,35747,moins de 500 lignes dans au moins un groupe ou...
1,1,10348,0,10348,moins de 500 lignes dans au moins un groupe ou...
2,2,10765,0,10765,moins de 500 lignes dans au moins un groupe ou...
3,40,4750,4367,383,moins de 500 lignes dans au moins un groupe ou...


{'ecart_global_points': np.float64(14.832), 'mediane_delai_avec_rappel': 11.0, 'mediane_delai_sans_rappel': 0.0}

QUESTION 14 : HYPOTHÈSE TESTÉE
{'verdict': 'contredite', 'suite': "Le gain mesuré est de 0.01 barre(s), comparé au seuil annoncé de +2 barres. Je conserve le résultat comme une mesure à vérifier sur un second découpage ; si l'hypothèse est contredite, je ne prétends pas que cette feature améliore le modèle et je cherche une autre représentation."}


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
